# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第13章 音韻変異とコーパス音韻論 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch13_worked_example.R`

対象章: 第13章「音韻変異とコーパス音韻論」通しの作例

「予測可能性・頻度・レマの効果」節で述べた、ジュラフスキーら[-@jurafsky2002]
およびガール[-@gahl2008]の知見(頻度・予測可能性が高い語ほど短く発音される)
を、架空のコーパスデータで再現する。実測データではない。

データ生成の設計:
  50語(レマ)を、対数頻度が一様に分布するように設定し、各レマについて
  20名の話者×平均4トークンの出現を生成する。真の効果として、
  log(頻度)が1単位上がるごとに持続時間が対数尺度で一定割合短くなる
  ように設定し(頻度効果)、文脈的な予測可能性(0-1)が高いほど
  さらに短縮するように設定する(予測可能性効果)。話者ごとの
  発話速度の個人差もランダム効果として加える。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

n_lemma    <- 50
n_speakers <- 20

lemmas <- tibble(
  lemma      = sprintf("lemma%02d", 1:n_lemma),
  log_freq   = runif(n_lemma, 0, 8),                 # 対数頻度(コーパス内トークン数の対数相当)
  lemma_re   = rnorm(n_lemma, 0, 0.05)                # レマ固有のランダム切片(対数尺度)
)

speakers <- tibble(
  speaker    = sprintf("sp%02d", 1:n_speakers),
  speaker_re = rnorm(n_speakers, 0, 0.08)             # 話者固有の発話速度差(対数尺度)
)

true_freq_effect <- -0.035   # log(頻度)1単位あたりの対数持続時間の変化
true_pred_effect <- -0.25    # 予測可能性(0-1)1単位あたりの対数持続時間の変化
base_log_dur      <- log(220) # 基準となる持続時間(約220ms)

d <- lemmas |>
  tidyr::crossing(speakers) |>
  rowwise() |>
  mutate(n_tok = rpois(1, 4) + 1) |>
  ungroup() |>
  tidyr::uncount(n_tok) |>
  mutate(
    predictability = runif(n(), 0, 1),
    log_dur_mean = base_log_dur +
      true_freq_effect * (log_freq - mean(log_freq)) +
      true_pred_effect * (predictability - 0.5) +
      lemma_re + speaker_re,
    log_duration = log_dur_mean + rnorm(n(), 0, 0.12),
    duration_ms  = exp(log_duration)
  )

cat("生成したトークン数:", nrow(d), " / レマ数:", n_lemma, " / 話者数:", n_speakers, "\n\n")

m <- lmer(log(duration_ms) ~ scale(log_freq, scale = FALSE) + predictability +
            (1 | speaker) + (1 | lemma), data = d)
print(summary(m))

fx <- fixef(m)
pct_per_logfreq <- (exp(fx["scale(log_freq, scale = FALSE)"]) - 1) * 100
pct_per_pred    <- (exp(fx["predictability"]) - 1) * 100

cat("\n対数頻度1単位あたりの持続時間の変化:", round(pct_per_logfreq, 2), "%\n")
cat("予測可能性が0から1に上がるときの持続時間の変化:", round(pct_per_pred, 2), "%\n")

p <- d |>
  mutate(freq_bin = cut(log_freq, breaks = 6)) |>
  group_by(freq_bin) |>
  summarise(mean_dur = mean(duration_ms), mean_logfreq = mean(log_freq), n = n(), .groups = "drop")

p1 <- ggplot(d, aes(log_freq, duration_ms)) +
  geom_point(alpha = 0.15, size = 0.8, colour = "#34518A") +
  geom_smooth(method = "lm", colour = "#A8681F", linewidth = 1, se = TRUE) +
  labs(x = "レマの対数頻度", y = "持続時間 (ms)",
       caption = "シミュレーションデータ。scripts/R/ch13_worked_example.R より生成。")
save_fig(p1, "fig13-01-frequency-duration", chapter = "ch13")

write_csv(d, here::here("data", "processed", "ch13_lemma_duration_sim.csv"))
cat("\n完了。図は assets/figures/ch13/ に保存した。\n")


## `ch13_large_n_problem.R`

対象章: 第13章「音韻変異とコーパス音韻論」

「大N問題」節の作例。理論的には無視できるほど小さな効果量でも、
サンプルサイズが極端に大きければ統計的に有意と判定されてしまうことを、
架空のデータで実際に確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(10)

trivial_effect_ms <- 0.5   # 理論的には無視できる、きわめて小さな効果量
sd_duration <- 30

for (n in c(100, 5000, 200000)) {
  a <- rnorm(n, 150, sd_duration)
  b <- rnorm(n, 150 + trivial_effect_ms, sd_duration)
  test <- t.test(b, a)
  cat("N =", n, "(各群): 観測された差 =", round(diff(test$estimate) * -1, 3),
      "ms, p =", format.pval(test$p.value, digits = 3), "\n")
}


## `ch13_speaker_bootstrap.R`

対象章: 第13章「音韻変異とコーパス音韻論」

「大N問題」節の作例。トークン単位でリサンプリングした場合と、話者
単位でリサンプリングした場合とで、効果量の不確実性(標準誤差)の
推定がどう異なるかを、話者間相関を持つ架空のコーパスデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_speakers <- 15
n_tokens_per_speaker <- 200

speakers <- tibble(speaker = sprintf("sp%02d", 1:n_speakers),
                    spk_re = rnorm(n_speakers, 0, 15))   # 話者ごとの安定した個人差

d <- speakers |>
  tidyr::crossing(token = 1:n_tokens_per_speaker) |>
  mutate(duration = 150 + spk_re + rnorm(n(), 0, 8))

n_boot <- 500

boot_token_level <- map_dbl(1:n_boot, function(i) {
  resampled <- d |> slice_sample(n = nrow(d), replace = TRUE)
  mean(resampled$duration)
})

boot_speaker_level <- map_dbl(1:n_boot, function(i) {
  resampled_speakers <- sample(speakers$speaker, n_speakers, replace = TRUE)
  resampled <- d |> filter(speaker %in% resampled_speakers)
  mean(resampled$duration)
})

cat("トークン単位のブートストラップによるSE:", round(sd(boot_token_level), 3), "\n")
cat("話者単位のブートストラップによるSE:", round(sd(boot_speaker_level), 3), "\n")
cat("SEの比率(話者単位/トークン単位):",
    round(sd(boot_speaker_level) / sd(boot_token_level), 1), "倍\n")
